# 🚀 MDAAR: Multi-role Dynamic Architecture for Automated Repositories
### AI-Driven Software Documentation Framework (Complete Interactive Pipeline)
---
**Key Architecture Highlights:**
1. **100% Free & Zero-Cost:** Runs entirely on Google Colab and persists state in Google Drive.
2. **Independent SSH Key Pairs:** Dedicated Deploy Keys for source code and separate Deploy Keys for external docs repositories.
3. **Multi-Project Documentation Hub:** External docs repo stores docs in a dedicated `<ProjectName>/` folder, allowing one repository to document multiple projects.
4. **Flexible AI Model Gateway:** Use Google Gemini API (with automated key rotation) or local offline open-source models (Transformers / GGUF).
5. **Automated Schema & Route Parsers:** SQL DDL ➔ Real Mermaid ERD diagrams, and source code ➔ API Reference tables.
6. **Multi-Role AI Agents:** Specialized agents for Architecture, Database, APIs, DevOps/Operations, User Guides, and QA.
7. **Continuous Living Documentation:** Automated `git diff` change detection to update only affected documentation.
8. **Interactive In-Memory AI Assistant (RAG):** Query project documentation in natural language with source citation.
9. **MkDocs Material + Cloudflare Tunnel:** Instant public interactive portal with dark mode and Mermaid support.

## 1️⃣ Environment Setup & Self-Installation
Mount Google Drive for persistent storage and install **MDAAR** directly from GitHub along with all dependencies (`mkdocs-material`, `google-genai`, `cloudflared`).

In [ ]:
#@title 1. Initialize Runtime, Mount Drive & Install MDAAR { display-mode: "form" }
import os, sys, json, shutil, subprocess
from pathlib import Path

# 1. Mount Google Drive for persistence
DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ Google Drive mounted successfully.')
    except Exception as e:
        print('⚠️ Drive mount notice (using local storage fallback):', e)

# 2. Install MDAAR framework directly from GitHub along with quantization tools (transformers, accelerate, bitsandbytes)
print('⏳ Installing MDAAR framework, dependencies, and quantization acceleration tools...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers', 'accelerate', 'bitsandbytes'], check=True)

# 3. Verify / Install cloudflared
cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ Installing cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

print('🎉 Environment initialized, MDAAR & quantization tools installed successfully!')

## 2️⃣ Source Repository Configuration (Git)
Configure your project name, source code Git repository URL, and authentication method.
If **SSH** is selected, a dedicated `id_ed25519_source` key pair is generated and the public Deploy Key is printed below.

In [ ]:
#@title 2. Configure Source Repository { display-mode: "form" }
PROJECT_NAME = "MySystem" #@param {type:"string"}
GIT_SOURCE_URL = "https://github.com/octocat/Hello-World.git" #@param {type:"string"}
SOURCE_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if os.path.exists('/content/drive/MyDrive'):
    BASE_WORKSPACE = Path('/content/drive/MyDrive/AI_Documentation_Workspace')
else:
    BASE_WORKSPACE = Path('/content/AI_Documentation_Workspace')

from mdaar_pipeline import MDAARPipeline, ModelGateway

pipeline = MDAARPipeline(
    project_name=PROJECT_NAME,
    base_workspace=BASE_WORKSPACE,
    git_source_url=GIT_SOURCE_URL,
    source_auth=SOURCE_AUTH_METHOD
)

if SOURCE_AUTH_METHOD == "SSH":
    source_pub_key = pipeline.setup_source_ssh()
    print('='*70)
    print('🔑 Source Repository SSH Public Key (Deploy Key):')
    print('Add this to your code repository settings (Settings -> Deploy Keys):')
    print(source_pub_key)
    print('='*70)

print(f'✅ Project workspace ready at:\n📁 {pipeline.project_dir}')

## 3️⃣ External Shared Documentation Repository (Optional)
Recommended: Maintain a private repository on GitHub/GitLab as your central documentation hub.
MDAAR will automatically save docs inside `projects/<ProjectName>/` allowing one repository to hold documentation for all your projects!
Uses an **independent SSH Deploy Key** (`id_ed25519_docs`).

In [ ]:
#@title 3. Configure External Documentation Repository { display-mode: "form" }
ENABLE_EXTERNAL_DOCS_REPO = False #@param {type:"boolean"}
GIT_DOCS_URL = "" #@param {type:"string"}
DOCS_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if ENABLE_EXTERNAL_DOCS_REPO and GIT_DOCS_URL.strip():
    pipeline.git_docs_url = GIT_DOCS_URL.strip()
    pipeline.docs_auth = DOCS_AUTH_METHOD
    
    if DOCS_AUTH_METHOD == "SSH":
        docs_pub_key = pipeline.setup_docs_ssh()
        print('='*70)
        print('🔑 Docs Repository SSH Public Key (Deploy Key):')
        print('Add this to your documentation repository (Settings -> Deploy Keys with Write access):')
        print(docs_pub_key)
        print('='*70)
    print(f'✅ External Documentation Repository linked: {GIT_DOCS_URL}')
else:
    pipeline.git_docs_url = None
    print('ℹ️ Documentation will be stored locally and permanently in Google Drive.')

## 4️⃣ AI Intelligence Layer (Model Gateway)
Select between **Google Gemini API** (fast, free tier with automated key rotation) or **High-Performance Local LLMs (4-bit NF4 Quantization)** running on Colab T4 GPU:
- **Qwen2.5-Coder-7B-Instruct:** SOTA open-weights code & architecture reasoning model (100% Free & Ungated, no token required).
- **Meta-Llama-3.1-8B-Instruct:** Exceptional instruction following (requires Hugging Face Token).
- **Mistral-7B-Instruct-v0.3:** Extended 32k context and precise technical reasoning (requires Hugging Face Token).
- **Gemma-2-9B-IT:** Google's state-of-the-art open-weights model (requires Hugging Face Token).
> 💡 **T4 GPU Optimization:** 4-bit NF4 quantization consumes only ~5.5 GB VRAM out of 15 GB, preventing crashes and out-of-memory errors entirely.

In [ ]:
#@title 4. Configure AI Model Provider { display-mode: "form" }
AI_PROVIDER = "Gemini" #@param ["Gemini", "Local-Offline"]
GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-1.5-pro", "gemini-2.0-flash"]
GEMINI_KEYS_INPUT = "" #@param {type:"string"}

# High-Performance Local Models (Optimized for Colab T4 GPU via 4-bit NF4 Quantization)
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
QUANTIZATION = "4bit" #@param ["4bit", "None"]
HF_TOKEN_INPUT = "" #@param {type:"string"}

# Extract Gemini keys
keys_list = [k.strip() for k in GEMINI_KEYS_INPUT.split(',') if k.strip()]
if not keys_list:
    try:
        from google.colab import userdata
        secret_key = userdata.get('GEMINI_API_KEY')
        if secret_key:
            keys_list.append(secret_key)
    except:
        pass

# Extract Hugging Face token for gated models (Llama, Mistral, Gemma)
hf_token = HF_TOKEN_INPUT.strip() or None
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except:
        pass

gateway = ModelGateway(
    provider=AI_PROVIDER,
    model_name=GEMINI_MODEL,
    api_keys=keys_list,
    local_model_path=LOCAL_MODEL_NAME,
    hf_token=hf_token,
    quantization=QUANTIZATION
)
pipeline.gateway = gateway

print(f'✅ Model Gateway configured successfully!')
print(f'  • Active Provider: {AI_PROVIDER}')
if AI_PROVIDER == 'Gemini':
    print(f'  • Model: {GEMINI_MODEL} (Registered keys for rotation: {len(keys_list)})')
else:
    print(f'  • Offline Local Model: {LOCAL_MODEL_NAME}')
    print(f'  • Quantization Mode: {QUANTIZATION}')

## 5️⃣ Code Synchronization & Living Documentation Tracker
Clone/pull repository and detect code modifications via `git diff` since the last documented commit.

In [ ]:
#@title 5. Sync Code and Detect Changes { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 Current Git Commit: ({current_commit[:7] if current_commit else "Initial"})')
if changed_files:
    print(f'🔍 Detected {len(changed_files)} changed files since last documentation run:')
    for f in changed_files[:10]:
        print('  •', f)
else:
    print('ℹ️ Clean working state or initial run.')

## 6️⃣ Documentation Generation & Multi-Agent Execution
Generate standard 4-package documentation structure, run automated SQL & API parsers, and execute specialized AI agents.

In [ ]:
#@title 6. Execute Multi-Agent Generation { display-mode: "form" }
FORCE_FULL_GENERATION = True #@param {type:"boolean"}

# 1. Initialize documentation tree and register
created_count = pipeline.init_framework_structure()
print(f'✅ Documentation tree initialized ({created_count} templates updated).')

# 2. Extract genuine schema and API routes
pipeline.extract_and_generate_specialized()

# 3. Run multi-role AI agents
target_changes = None if FORCE_FULL_GENERATION else changed_files
pipeline.run_multi_agent_generation(changed_files=target_changes)

# 4. Record state
if current_commit:
    pipeline.save_state(current_commit)
    print(f'💾 Saved documentation state for commit ({current_commit[:7]}).')

## 7️⃣ Documentation Portal & Cloudflare Tunnel
Build and launch interactive MkDocs Material site and generate an instant public tunnel URL.

In [ ]:
#@title 7. Launch Portal and Push to Remote Docs Repo { display-mode: "form" }
tunnel_url = pipeline.serve_and_tunnel(port=8000)

if tunnel_url:
    print('='*70)
    print('🎉 Interactive Documentation Portal is live at:')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ Server running locally, check tunnel logs.')

# Sync with external Git documentation repository if configured
sync_docs_res = pipeline.sync_docs_repo()
print('📡 Remote Docs Sync:', sync_docs_res)

## 8️⃣ Natural Language Q&A Assistant (RAG over Docs)
Query your generated project documentation in natural language with source citation.

In [ ]:
#@title 8. Query Project Documentation { display-mode: "form" }
QUESTION = "What is the system architecture and core components?" #@param {type:"string"}
AUDIENCE = "Developer" #@param ["Developer", "Management", "End-User"]

chat_engine = pipeline.get_chat_engine()
chat_engine.load_documents()

print(f'🔍 Retrieving knowledge and generating answer for: "{QUESTION}"...\n')
answer = chat_engine.ask(QUESTION, audience=AUDIENCE)
print('='*70)
print(answer)
print('='*70)